In [3]:
import pandas as pd
import numpy as np
import mlflow, mlflow.sklearn
import joblib, os, warnings
warnings.filterwarnings('ignore')
 
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute          import SimpleImputer
from sklearn.linear_model    import LogisticRegression, LinearRegression
from sklearn.ensemble        import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics         import (accuracy_score, precision_score, recall_score,
                                     f1_score, roc_auc_score,
                                     mean_squared_error, mean_absolute_error, r2_score)
from xgboost import XGBClassifier, XGBRegressor
 
os.makedirs("models", exist_ok=True)

In [4]:
# MLflow setup

mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("RealEstate_InvestmentAdvisor")

2026/04/20 19:06:12 INFO mlflow.tracking.fluent: Experiment with name 'RealEstate_InvestmentAdvisor' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:C:/Users/pd413/data science/New project/mlruns/125846045408259033', creation_time=1776692172363, experiment_id='125846045408259033', last_update_time=1776692172363, lifecycle_stage='active', name='RealEstate_InvestmentAdvisor', tags={}, trace_location=None, workspace='default'>

In [5]:
# load data
df    = pd.read_csv("cleaned_data.csv")
X     = df.drop(columns=['Good_Investment','Future_Price_5Y'], errors='ignore')
y_clf = df['Good_Investment']
y_reg = df['Future_Price_5Y']
 
imputer = SimpleImputer(strategy='median')
X       = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
 
X_train, X_test, yc_train, yc_test = train_test_split(
    X, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
_,       _,      yr_train, yr_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42)

In [6]:
# Classification
print("Logging Classification experiments...")
 
clf_configs = [
    ("LogisticRegression", LogisticRegression(max_iter=1000, random_state=42),
     {"max_iter":1000, "solver":"lbfgs"}),
    ("RandomForest_Clf",   RandomForestClassifier(n_estimators=100, random_state=42),
     {"n_estimators":100, "max_depth":"None"}),
    ("XGBoost_Clf",        XGBClassifier(n_estimators=100, learning_rate=0.1,
                                         max_depth=6, random_state=42,
                                         eval_metric='logloss', verbosity=0),
     {"n_estimators":100, "learning_rate":0.1, "max_depth":6}),
]
 
best_clf, best_clf_f1, best_clf_name = None, -1, ""
 
for name, model, params in clf_configs:
    with mlflow.start_run(run_name=name):
        model.fit(X_train, yc_train)
        yp    = model.predict(X_test)
        yprob = model.predict_proba(X_test)[:,1]
 
        f1  = f1_score       (yc_test, yp)
        auc = roc_auc_score  (yc_test, yprob)
        acc = accuracy_score (yc_test, yp)
        cv  = cross_val_score(model, X, y_clf, cv=5, scoring='accuracy').mean()
 
        mlflow.log_params(params)
        mlflow.log_metrics({"accuracy":acc,"f1_score":f1,"roc_auc":auc,"cv_accuracy":cv})
        mlflow.set_tag("task", "classification")
        mlflow.sklearn.log_model(model, name="model",
                                 registered_model_name=name)
        print(f"  {name}: F1={f1:.4f}  AUC={auc:.4f}")
 
        if f1 > best_clf_f1:
            best_clf_f1, best_clf, best_clf_name = f1, model, name
 
joblib.dump(best_clf, "models/classifier.pkl")
print(f"★ Best CLF: {best_clf_name} (F1={best_clf_f1:.4f})")

Logging Classification experiments...


2026/04/20 19:08:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'LogisticRegression'.
Created version '1' of model 'LogisticRegression'.


  LogisticRegression: F1=0.9697  AUC=0.9982


2026/04/20 19:11:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'RandomForest_Clf'.
Created version '1' of model 'RandomForest_Clf'.


  RandomForest_Clf: F1=0.9984  AUC=1.0000


2026/04/20 19:11:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  XGBoost_Clf: F1=0.9984  AUC=1.0000
★ Best CLF: RandomForest_Clf (F1=0.9984)


Successfully registered model 'XGBoost_Clf'.
Created version '1' of model 'XGBoost_Clf'.


In [8]:
# Regression
print("\nLogging Regression experiments...")
 
reg_configs = [
    ("LinearRegression", LinearRegression(),
     {"fit_intercept": True}),
    ("RandomForest_Reg", RandomForestRegressor(n_estimators=50, random_state=42),  # 100→50
     {"n_estimators":50, "max_depth":"None"}),
    ("XGBoost_Reg",      XGBRegressor(n_estimators=50, learning_rate=0.1,         # 100→50
                                      max_depth=6, random_state=42, verbosity=0),
     {"n_estimators":50, "learning_rate":0.1, "max_depth":6}),
]

best_reg, best_reg_rmse, best_reg_name = None, float('inf'), ""

for name, model, params in reg_configs:
    with mlflow.start_run(run_name=name):
        model.fit(X_train, yr_train)
        yp    = model.predict(X_test)
        rmse  = float(np.sqrt(mean_squared_error(yr_test, yp)))
        mae   = float(mean_absolute_error(yr_test, yp))
        r2    = float(r2_score(yr_test, yp))
        cv_r2 = float(cross_val_score(model, X, y_reg, cv=3, scoring='r2').mean())  # 5→3

        mlflow.log_params(params)
        mlflow.log_metrics({"rmse":rmse, "mae":mae, "r2":r2, "cv_r2":cv_r2})
        mlflow.set_tag("task", "regression")
        mlflow.sklearn.log_model(model, name="model",
                                 registered_model_name=name)
        print(f"  {name}: RMSE={rmse:.4f}  R²={r2:.4f}  CV-R²={cv_r2:.4f}")

        if rmse < best_reg_rmse:
            best_reg_rmse, best_reg, best_reg_name = rmse, model, name

joblib.dump(best_reg,        "models/regressor.pkl")
joblib.dump(list(X.columns), "models/feature_names.pkl")
joblib.dump(imputer,         "models/imputer.pkl")
print(f"\n★ Best REG: {best_reg_name} (RMSE={best_reg_rmse:.4f})")
print("\n✅ Step 4 Done!")
print("   View UI: mlflow ui  →  http://localhost:5000")


Logging Regression experiments...


2026/04/20 19:34:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'LinearRegression' already exists. Creating a new version of this model...
Created version '2' of model 'LinearRegression'.


  LinearRegression: RMSE=210.0788  R²=-0.0001  CV-R²=0.9936


2026/04/20 19:42:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'RandomForest_Reg' already exists. Creating a new version of this model...
Created version '2' of model 'RandomForest_Reg'.


  RandomForest_Reg: RMSE=212.9651  R²=-0.0277  CV-R²=1.0000


2026/04/20 19:42:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  XGBoost_Reg: RMSE=210.2081  R²=-0.0013  CV-R²=1.0000

★ Best REG: LinearRegression (RMSE=210.0788)

✅ Step 4 Done!
   View UI: mlflow ui  →  http://localhost:5000


Registered model 'XGBoost_Reg' already exists. Creating a new version of this model...
Created version '2' of model 'XGBoost_Reg'.
